# D165 — Introduction to MySQL Ranking Functions

Ranking functions place rows in an order without combining them. This lesson uses ten simple student scores. Some scores are tied so that the differences between ranking functions are easy to see.

## 1. Connect and prepare a small table

A temporary table exists only for the current MySQL connection. It disappears when the connection closes.

In [ ]:
import os
import mysql.connector

connection = mysql.connector.connect(
    host=os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USERNAME', 'root'),
    password=os.environ.get('MYSQL_PASSWORD', 'root'),
    database=os.environ.get('MYSQL_DATABASE', 'olist_import_lab'),
)
print('Connected:', connection.is_connected())

In [ ]:
def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if not cursor.with_rows:
            connection.commit()
            print(f'Statement completed. Affected rows: {cursor.rowcount}')
            return []
        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchall()
        values = [[str(value) for value in row] for row in rows]
        widths = [len(column) for column in columns]
        for row in values:
            widths = [max(width, len(value)) for width, value in zip(widths, row)]
        print(' | '.join(c.ljust(w) for c, w in zip(columns, widths)))
        print('-+-'.join('-' * w for w in widths))
        for row in values:
            print(' | '.join(v.ljust(w) for v, w in zip(row, widths)))
        return rows
    finally:
        cursor.close()

In [ ]:
execute_sql('DROP TEMPORARY TABLE IF EXISTS ranking_demo_scores')
execute_sql("""
CREATE TEMPORARY TABLE ranking_demo_scores (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(20) NOT NULL,
    learning_track VARCHAR(20) NOT NULL,
    score INT NOT NULL
)
""")
execute_sql("""
INSERT INTO ranking_demo_scores VALUES
(1, 'Asha',   'Data',  95),
(2, 'Bilal',  'Data',  90),
(3, 'Chen',   'Data',  90),
(4, 'Divya',  'Data',  80),
(5, 'Eshan',  'Data',  75),
(6, 'Fatima', 'Cloud', 95),
(7, 'Gopal',  'Cloud', 88),
(8, 'Hana',   'Cloud', 88),
(9, 'Ishan',  'Cloud', 70),
(10,'Jaya',   'Cloud', 60)
""")

In [ ]:
execute_sql("""
SELECT * FROM ranking_demo_scores
ORDER BY score DESC, student_id
""")

## 2. The window used for ranking

Ranking functions use `OVER`. The `ORDER BY` inside `OVER` decides the ranking order.

```sql
RANK() OVER (ORDER BY score DESC)
```

`DESC` means the highest score comes first. The outer `ORDER BY` controls display order; it does not calculate the rank.

## 3. `ROW_NUMBER`, `RANK`, and `DENSE_RANK` together

- `ROW_NUMBER` gives every row a different sequence number. Ties are broken by the extra `student_id` ordering.
- `RANK` gives tied scores the same rank and leaves a gap after the tie. Scores 90 and 90 receive rank 3, so the next rank is 5.
- `DENSE_RANK` gives tied scores the same rank but does not leave a gap. The rank after 3 is 4.

In [ ]:
execute_sql("""
SELECT student_name, learning_track, score,
       ROW_NUMBER() OVER (ORDER BY score DESC, student_id) AS row_number_value,
       RANK()       OVER (ORDER BY score DESC) AS rank_value,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank_value
FROM ranking_demo_scores
ORDER BY score DESC, student_id
""")

## 4. `NTILE` creates buckets

`NTILE(4)` divides the ordered rows into four groups with nearly equal numbers of rows. These groups are often called quartiles. When 10 rows are split into 4 buckets, the first buckets receive an extra row. `NTILE` is based on row position, not equal score ranges.

In [ ]:
execute_sql("""
SELECT student_name, score,
       NTILE(4) OVER (ORDER BY score DESC, student_id) AS score_bucket
FROM ranking_demo_scores
ORDER BY score DESC, student_id
""")

## 5. Rank separately inside each track

`PARTITION BY` starts a new ranking for each group. It does not remove rows. Here, Data and Cloud each receive ranks beginning at 1.

In [ ]:
execute_sql("""
SELECT student_name, learning_track, score,
       RANK() OVER (PARTITION BY learning_track ORDER BY score DESC) AS track_rank
FROM ranking_demo_scores
ORDER BY learning_track, track_rank, student_id
""")

## 6. Keep the top two from each track

MySQL cannot filter a window-function result in the same query level's `WHERE`. A CTE calculates `ROW_NUMBER` first, and the outer query filters it. `ROW_NUMBER` returns exactly two rows per track. Using `RANK` instead could return more than two when the second score is tied.

In [ ]:
execute_sql("""
WITH ranked_students AS (
    SELECT student_name, learning_track, score,
           ROW_NUMBER() OVER (
               PARTITION BY learning_track
               ORDER BY score DESC, student_id
           ) AS track_row_number
    FROM ranking_demo_scores
)
SELECT * FROM ranked_students
WHERE track_row_number <= 2
ORDER BY learning_track, track_row_number
""")

## 7. Choosing a ranking function

| Need | Function |
|---|---|
| A unique sequence for every row | `ROW_NUMBER` |
| Competition ranking with gaps after ties | `RANK` |
| Ranking without gaps after ties | `DENSE_RANK` |
| A fixed number of similarly sized row buckets | `NTILE` |

Always include enough columns in the window's `ORDER BY` when a stable tie-break is required. `PERCENT_RANK` and `CUME_DIST` are explained separately in D166 and D167.

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed; the temporary table is gone.')